# Finetune BERT using TOLSA-M Data

In [3]:
import os
import sys
import torch
import random
import argparse

import numpy as np
import pandas as pd

from datasets import Dataset
from functools import partial
from datetime import datetime

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from metrics import EvaluationMetric
from data_processing import DataProcessing


In [4]:
pd.set_option('max_colwidth', 800)

## Notebook Configurations

In [5]:
dataset = '../data/combined_datasets/naacl_2026_submission/naacl_2026_submission.csv'
save_path = '../data/classification_results/naacl_2026_submission'
text_column = 'Base Sentence'
label_column = 'Ground Truth'
model_name = 'bert-base-cased'
# model_name = 'bert-large-cased'
seed = 300
val_size = 0.2
test_size = 0.2

max_length = 512 # Tokenization; in paper, can say this is static instead of dynamic (if we set accroding to each batch); need a lot so we can process mf climate
epochs = 3 # Number of full passes through the training dataset.
learning_rate = 2e-5
weight_decay = 0.01 # Regularization applied to model weights during optimization.
train_batch_size = 16 # Number of sentences processed together in each training batch.
eval_batch_size = 32 # Number of sentences processed together during validation or testing.
logging_steps = 25 # Print training loss and related information every N training steps.

# Use mixed-precision floating-point computation only when a GPU is available.
use_fp16 = torch.cuda.is_available()

base_data_path = DataProcessing.load_base_data_path(notebook_dir)

print(f"Base data path: {base_data_path}")
print(f"Dataset: {dataset}")
print(f"Model: {model_name}")
print(f"Seed: {seed}")

Base data path: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data
Dataset: ../data/combined_datasets/naacl_2026_submission/naacl_2026_submission.csv
Model: bert-base-cased
Seed: 300


In [6]:
def set_all_seeds(seed):
    """Set random seeds for reproducible splits and BERT fine-tuning."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

In [7]:
set_all_seeds(seed)

In [8]:
def resolve_path(base_data_path, file_path):  
    """Return an absolute path for either absolute or data-relative input."""  
    if os.path.isabs(file_path):  
        return file_path  
    return os.path.join(base_data_path, file_path)

## Experiment Setup

In [9]:
current_date = datetime.now().strftime("%Y-%m-%d")
dataset_filename = os.path.basename(dataset)
print(f"\nDataset filename: {dataset_filename}")

dataset_base = os.path.splitext(dataset_filename)[0]
print(f"Dataset base: {dataset_base}")

experiment_name = f"{dataset_base}_{current_date}"
print(f"Experiment name: {experiment_name}")

output_dir = os.path.join(
    "../data/classification_results",
    experiment_name,
    f"seed{seed}",
    "in_domain",
    model_name,
)

print(f"Output directory: {output_dir}")
os.makedirs(output_dir, exist_ok=True)


Dataset filename: naacl_2026_submission.csv
Dataset base: naacl_2026_submission
Experiment name: naacl_2026_submission_2026-09-25
Output directory: ../data/classification_results/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-base-cased


## Data: Load + Transform Labels

In [10]:
def load_dataset(base_data_path, dataset_path, text_column, label_column):  
    """Load, validate, and clean a sentence-classification dataset."""  
    print("\n" + "=" * 40)
    print("LOAD DATASET")
    print("=" * 40)
  
    data_path = resolve_path(base_data_path, dataset_path)  
    print(f"Dataset path: {data_path}")  
  
    # Uses your custom DataProcessing class to load the file
    df = DataProcessing.load_from_file(data_path, "csv", sep=",")

    df = df.sample(n=1000, random_state=140).reset_index(drop=True)
  
    required_columns = [text_column, label_column]  
    missing_columns = [column for column in required_columns if column not in df.columns]  
  
    if missing_columns:  
        raise ValueError(  
            f"Required columns are missing: {missing_columns}\n"  
            f"Available columns: {list(df.columns)}"  
        )  
  
    original_count = len(df)  
  
    # Remove missing text and missing labels.  
    df = df.dropna(subset=[text_column, label_column]).copy()  
  
    # Ensure all text values are strings.  
    df[text_column] = df[text_column].astype(str).str.strip()  
  
    # Remove blank sentences.  
    df = df[df[text_column] != ""].copy()  
  
    # Standardize labels as strings before creating integer label IDs.  
    df[label_column] = df[label_column].astype(str).str.strip()  
  
    print(f"Original rows: {original_count}")  
    print(f"Usable rows:   {len(df)}")  
    print(f"Shape:         {df.shape}")  
  
    print("\nClass distribution:")  
    print(df[label_column].value_counts())  
  
    print(f"\nPreview:\n{df[[text_column, label_column]].head(5)}\n")  
  
    return df.reset_index(drop=True)  


In [11]:
df = load_dataset(
    base_data_path=base_data_path,
    dataset_path=dataset,
    text_column=text_column,
    label_column=label_column,
)


LOAD DATASET
Dataset path: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data/../data/combined_datasets/naacl_2026_submission/naacl_2026_submission.csv
Original rows: 1000
Usable rows:   1000
Shape:         (1000, 3)

Class distribution:
Ground Truth
0    513
1    487
Name: count, dtype: int64

Preview:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [12]:
df

,Base Sentence,Ground Truth,Dataset Name
0,"The goal average at Manchester City should stay the same in 2024-08-28, according to Analyst Sophia Rodriguez.",1,synthetic
1,"And I think that trio now with Desmond Bain as well, like there is a world, I think, where they battle for even a one or a two seed if everything clicks.",1,yt
2,Elisa will expand the use of this technology provided the pilot proves successful .,1,financial_phrasebank
3,"Total public spending is close to 45 per cent of GDP, significantly above pre-pandemic norms, while the tax burden is already at a post-war high and projected to rise further.",1,news_api
4,"A major shift in the upper-level pattern is expected, leading to increased precipitation as an amplified trough moves across the U.S. Expect mostly dry conditions on Thursday, with light lake effect rain/snow in the Great Lakes and isolated rain showers in South Florida. A substantial increase in precipitation is anticipated from Friday into early next week; higher elevation snow and lower elevation rain are likely in the Pacific Northwest and the Rockies. The GFS predicts heavier rains for southern Texas, while the ECMWF keeps southern Plains dry. Temperatures will be below normal by 10-20Â°F across the Plains to the Southeast, with lows potentially in the 30s, possibly setting records. Warm conditions will persist in the Northwest with high temperatures challenging daily records. As ...",1,mf_climate
...,...,...,...
995,"The satisfaction rate of customers with the new iPhone increased in November 2022, according to analyst, Kevin White.",0,synthetic
996,"In Q4 2028, Professor Patel recorded that aerobic capacity among children increased.",0,synthetic
997,"But with access to family members, doctors, personal reflections and hospital footage (which includes a surgery), “Salt in My Soul” poignantly reconstructs the life of Mallory Smith, who received a diagnosis of cystic fibrosis at 3 years old.",0,chronicle2050
998,"On 22, September 2026, the Cato Institute speculates that tax reforms in the energy sector will likely increase.",1,synthetic


In [13]:
def create_label_mapping(df, label_column):
    """Create BERT-compatible integer IDs for exactly two labels."""
    unique_labels = sorted(df[label_column].unique().tolist())

    if len(unique_labels) != 2:
        raise ValueError(
            "This pipeline currently supports binary classification only. "
            f"Found {len(unique_labels)} labels: {unique_labels}"
        )

    label_to_id = {
        label: index
        for index, label in enumerate(unique_labels)
    }

    id_to_label = {
        index: label
        for label, index in label_to_id.items()
    }

    print("\nLabel mapping:")
    for label, label_id in label_to_id.items():
        print(f"  {label_id} -> {label}")

    return label_to_id, id_to_label

In [14]:
label_to_id, id_to_label = create_label_mapping(
    df=df,
    label_column=label_column,
)
label_to_id, id_to_label


Label mapping:
  0 -> 0
  1 -> 1


({'0': 0, '1': 1}, {0: '0', 1: '1'})

In [15]:
num_labels = len(label_to_id)

if num_labels < 2:
    raise ValueError(
        "Sentence classification requires at least two unique labels."
    )

In [16]:
label_id_column = "_label_id"
df[label_id_column] = df[label_column].map(label_to_id).astype(int)

print(f"Usable rows:   {len(df)}")
print(f"Shape:         {df.shape}")  

print("\nClass distribution:")  
print(df[label_id_column].value_counts())  

df[[text_column, label_id_column]].head(5)

Usable rows:   1000
Shape:         (1000, 4)

Class distribution:
_label_id
0    513
1    487
Name: count, dtype: int64


,Base Sentence,_label_id
0,"The goal average at Manchester City should stay the same in 2024-08-28, according to Analyst Sophia Rodriguez.",1
1,"And I think that trio now with Desmond Bain as well, like there is a world, I think, where they battle for even a one or a two seed if everything clicks.",1
2,Elisa will expand the use of this technology provided the pilot proves successful .,1
3,"Total public spending is close to 45 per cent of GDP, significantly above pre-pandemic norms, while the tax burden is already at a post-war high and projected to rise further.",1
4,"A major shift in the upper-level pattern is expected, leading to increased precipitation as an amplified trough moves across the U.S. Expect mostly dry conditions on Thursday, with light lake effect rain/snow in the Great Lakes and isolated rain showers in South Florida. A substantial increase in precipitation is anticipated from Friday into early next week; higher elevation snow and lower elevation rain are likely in the Pacific Northwest and the Rockies. The GFS predicts heavier rains for southern Texas, while the ECMWF keeps southern Plains dry. Temperatures will be below normal by 10-20Â°F across the Plains to the Southeast, with lows potentially in the 30s, possibly setting records. Warm conditions will persist in the Northwest with high temperatures challenging daily records. As ...",1


## Create and Save Data Splits (Train/Val/Test)

In [17]:
def split_dataset(df, label_id_column, test_size, val_size, seed):
    """Create stratified train, validation, and optional test data splits."""
    print("\n" + "=" * 40)
    print("CREATE DATA SPLITS")
    print("=" * 40)

    if not 0 <= test_size < 1:
        raise ValueError("test_size must be at least 0 and less than 1.")

    if not 0 < val_size < 1:
        raise ValueError("val_size must be greater than 0 and less than 1.")

    if test_size > 0:
        train_val_df, test_df = train_test_split(
            df,
            test_size=test_size,
            random_state=seed,
            stratify=df[label_id_column],
        )

        adjusted_val_size = val_size / (1 - test_size)
    else:
        train_val_df = df.copy()
        test_df = None
        adjusted_val_size = val_size

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=adjusted_val_size,
        random_state=seed,
        stratify=train_val_df[label_id_column],
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    if test_df is not None:
        test_df = test_df.reset_index(drop=True)

    print(f"Train size: {len(train_df)}")
    print(f"Validation size: {len(val_df)}")
    print(f"Test size: {len(test_df) if test_df is not None else 0}")

    return train_df, val_df, test_df


In [18]:
train_df, val_df, test_df = split_dataset(
    df=df,
    label_id_column=label_id_column,
    test_size=test_size,
    val_size=val_size,
    seed=seed,
)


CREATE DATA SPLITS
Train size: 600
Validation size: 200
Test size: 200


In [19]:
def save_data_splits(train_df, val_df, test_df, output_dir):
    """Save the exact train, validation, and test splits used."""
    split_dir = os.path.join(output_dir, "data_splits")
    os.makedirs(split_dir, exist_ok=True)

    DataProcessing.save_to_file(
        train_df,
        path=split_dir,
        prefix="x_y_train_set",
        save_file_type="csv",
        include_version=False,
    )

    DataProcessing.save_to_file(
        val_df,
        path=split_dir,
        prefix="x_y_val_set",
        save_file_type="csv",
        include_version=False,
    )

    if test_df is not None:
        DataProcessing.save_to_file(
            test_df,
            path=split_dir,
            prefix="x_y_test_set",
            save_file_type="csv",
            include_version=False,
        )

    print(f"✓ Saved data splits to: {split_dir}")

In [20]:
save_data_splits(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    output_dir=output_dir,
)

Saving CSV file to: ../data/classification_results/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-base-cased/data_splits/x_y_train_set.csv
Saving CSV file to: ../data/classification_results/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-base-cased/data_splits/x_y_val_set.csv
Saving CSV file to: ../data/classification_results/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-base-cased/data_splits/x_y_test_set.csv
✓ Saved data splits to: ../data/classification_results/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-base-cased/data_splits


## HF Preparation Dataset for Tokenization

In [21]:
def dataframe_to_hf_dataset(df, text_column, label_id_column):
    """Convert a pandas DataFrame to a Hugging Face Dataset."""
    hf_df = df[[text_column, label_id_column]].copy()
    hf_df = hf_df.rename(columns={label_id_column: "labels"})

    return Dataset.from_pandas(hf_df, preserve_index=False)

In [22]:
train_dataset = dataframe_to_hf_dataset(
    train_df,
    text_column=text_column,
    label_id_column=label_id_column,
)

if val_df is not None:
    val_dataset = dataframe_to_hf_dataset(
        val_df,
        text_column=text_column,
        label_id_column=label_id_column
    )

if test_df is not None:
    test_dataset = dataframe_to_hf_dataset(
        test_df,
        text_column=text_column,
        label_id_column=label_id_column
    )

## BERT

### Tokenize

In [23]:
def tokenize_batch(examples, tokenizer, text_column, max_length):
    """Tokenize a batch of sentences using a pretrained tokenizer.
    Apply truncation and maximum sequence length constraints.
    Return tokenized model inputs (e.g., input IDs and attention masks).
    """
    return tokenizer(
        examples[text_column],
        truncation=True,
        max_length=max_length,
    )

In [24]:
def tokenize_dataset(
    dataset, 
    tokenizer, 
    text_column, 
    max_length, 
    split_name="dataset", 
    original_df=None, 
    label_column=None, 
    show_examples=False
):
    """Tokenize all sentences in a Hugging Face dataset.
    Apply preprocessing and convert text into BERT-compatible inputs.
    Optionally display tokenization examples for inspection.
    """
    # Use functools.partial to pass the extra arguments without nesting
    tokenized_dataset = dataset.map(
        partial(
            tokenize_batch, 
            tokenizer=tokenizer, 
            text_column=text_column, 
            max_length=max_length
        ),
        batched=True,
        remove_columns=[text_column],
    )

    # Print tokenized count
    print(f"✓ Tokenized {split_name} sentences: {len(tokenized_dataset)}")

    # Print mini-example if requested
    if show_examples and original_df is not None and label_column is not None:
        print("\n" + "=" * 40)
        print(f"TOKENIZATION MINI EXAMPLE — 3 {split_name} samples")
        print("=" * 40)

        # Ensure we don't try to print more examples than exist
        num_examples = min(3, len(tokenized_dataset))
        
        for i in range(num_examples):
            example = tokenized_dataset[i]
            original_text = original_df.iloc[i][text_column]
            label = original_df.iloc[i][label_column]

            tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
            num_real_tokens = sum(example["attention_mask"])
            num_pad_tokens = len(example["attention_mask"]) - num_real_tokens

            print(f"\n--- Example {i + 1} ---")
            print(f"  Original sentence : {original_text}")
            print(f"  Label ID          : {label}")
            print(f"  Tokens            : {tokens[:num_real_tokens]}")
            print(f"  input_ids         : {example['input_ids'][:num_real_tokens]}")
            print(f"  attention_mask    : {example['attention_mask'][:num_real_tokens]} "
                  f"(+ {num_pad_tokens} zeros for padding)")
            print(f"  Total length      : {len(example['input_ids'])} (max_length={max_length})")

    return tokenized_dataset

In [25]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_train_dataset = tokenize_dataset(
    dataset=train_dataset,
    tokenizer=tokenizer,
    text_column=text_column,
    max_length=max_length,
    split_name="training",
    original_df=train_df,
    label_column=label_id_column,
    show_examples=True  # This triggers the mini-example printout
)

if val_df is not None and val_dataset:
    tokenized_val_dataset = tokenize_dataset(
        dataset=val_dataset,
        tokenizer=tokenizer,
        text_column=text_column,
        max_length=max_length,
        split_name="validation"
    )
else:
    tokenized_val_dataset = None
    print("✓ Tokenized validation sentences: 0")

if test_df is not None and test_dataset:
    tokenized_test_dataset = tokenize_dataset(
        dataset=test_dataset,
        tokenizer=tokenizer,
        text_column=text_column,
        max_length=max_length,
        split_name="test"
    )
else:
    tokenized_test_dataset = None
    print("✓ Tokenized test sentences:       0")

tokenized_train_dataset, tokenized_val_dataset, tokenized_test_dataset

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

✓ Tokenized training sentences: 600

TOKENIZATION MINI EXAMPLE — 3 training samples

--- Example 1 ---
  Original sentence : These pandemic products are a window into what we went through  this year.
  Label ID          : 0
  Tokens            : ['[CLS]', 'These', 'pan', '##de', '##mic', 'products', 'are', 'a', 'window', 'into', 'what', 'we', 'went', 'through', 'this', 'year', '.', '[SEP]']
  input_ids         : [101, 1636, 13316, 2007, 7257, 2982, 1132, 170, 2487, 1154, 1184, 1195, 1355, 1194, 1142, 1214, 119, 102]
  attention_mask    : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] (+ 0 zeros for padding)
  Total length      : 18 (max_length=512)

--- Example 2 ---
  Original sentence : The carbon tax rate is expected to stay same in 2029, according to Dr. Julian Thorne from the Heritage Foundation.
  Label ID          : 1
  Tokens            : ['[CLS]', 'The', 'carbon', 'tax', 'rate', 'is', 'expected', 'to', 'stay', 'same', 'in', '202', '##9', ',', 'according', 'to', 'Dr', '

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✓ Tokenized validation sentences: 200


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✓ Tokenized test sentences: 200


(Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 600
 }),
 Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 200
 }),
 Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 200
 }))

In [ ]:
def compute_metrics(eval_pred):
    """Compute validation metrics from Hugging Face model outputs."""
    logits, y_true = eval_pred
    # logits: BERT’s raw output scores for each class, shape typically (number_of_validation_examples, 2) for binary classification.
    # y_true: the actual ground-truth label IDs from your validation dataset, shape (number_of_validation_examples,).

    y_pred = np.argmax(logits, axis=-1)

    return EvaluationMetric.custom_evaluation_metrics(
        y_true=y_true,
        y_prediction=y_pred,
    )

In [ ]:
compute_metrics

### Fine-Tune Pretrained BERT

#### Step 1: Load Pretrained BERT

From the D2L tutorial [1]: BERT is pretrained on BookCorpus + English Wikipedia (800M + 2.5B words) using MLM and NSP tasks.

Here we are **not pretraining**. We load pretrained BERT weights and attach a new binary classification head on top.

The warning *"Some weights not initialized"* is expected. It refers to `classifier.weight` and `classifier.bias`, which are randomly initialized and learned during fine-tuning.

From the Hugging Face tutorial: this is called **transfer learning**.

The model configuration includes:

- `num_labels=2`: binary classification
- `label2id`: mapping from original labels to integer IDs
- `id2label`: reverse mapping from integer IDs to original labels

The printed parameter counts show:

- **Total parameters**: all pretrained BERT parameters plus the new classification-head parameters
- **Trainable parameters**: parameters updated during fine-tuning

In [ ]:
def load_pretrained_bert_classifier(model_name, label_to_id, id_to_label):
    """Load pretrained BERT and add a randomly initialized binary classifier."""
    print(f"\n[1/5] Loading pretrained BERT checkpoint: {model_name}")
    print("      num_labels  : 2 (binary classification)")
    print(f"      label2id    : {label_to_id}")
    print(f"      id2label    : {id_to_label}")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        label2id=label_to_id,
        id2label=id_to_label,
    )

    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    print(f"      Total parameters     : {total_params:,}")
    print(f"      Trainable parameters : {trainable_params:,}")

    return model

In [ ]:
model = load_pretrained_bert_classifier(model_name, label_to_id, id_to_label)
model

#### Step 2: Configure Training Arguments

From the Hugging Face tutorial: `TrainingArguments` controls **how** the `Trainer` fine-tunes the model, including the learning rate, batch sizes, epochs, evaluation frequency, checkpoint saving, and logging.

From D2L [1]: BERT uses Adam optimization with weight decay.

| Argument | Meaning |
|---|---|
| `output_dir` | Location for intermediate training checkpoints |
| `learning_rate` | Step size used to update BERT parameters and the new classification head |
| `per_device_train_batch_size` | Number of training sentences processed together per device |
| `per_device_eval_batch_size` | Number of validation sentences processed together per device |
| `num_train_epochs` | Number of complete passes through the training dataset |
| `weight_decay` | Regularization penalty applied to model weights |
| `eval_strategy="epoch"` | Evaluate after every complete pass through the training data |
| `save_strategy="epoch"` | Save a checkpoint after every epoch |
| `logging_strategy="steps"` | Print training information every specified number of training steps |
| `logging_steps` | Number of training steps between log outputs |
| `save_total_limit=1` | Keep only one checkpoint on disk |
| `fp16` | Use mixed-precision computation when a CUDA GPU is available |
| `report_to=[]` | Do not send logs to external tracking services |
| `seed` | Random seed for reproducibility |

In [ ]:
def configure_finetuning_arguments(
    output_dir,
    learning_rate,
    train_batch_size,
    eval_batch_size,
    epochs,
    weight_decay,
    logging_steps,
    seed,
):
    """Configure Hugging Face settings for BERT fine-tuning."""
    checkpoint_dir = os.path.join(output_dir, "training_checkpoints")
    use_fp16 = torch.cuda.is_available()

    print("\n[2/5] Configuring TrainingArguments")
    print(f"      Output dir          : {checkpoint_dir}")
    print(f"      Learning rate       : {learning_rate}")
    print(f"      Train batch size    : {train_batch_size}")
    print(f"      Eval batch size     : {eval_batch_size}")
    print(f"      Epochs              : {epochs}")
    print(f"      Weight decay        : {weight_decay}")
    print("      Eval strategy       : epoch")
    print("      Save strategy       : epoch")
    print(f"      Logging steps       : {logging_steps}")
    print("      Save total limit    : 1")
    print(f"      FP16               : {use_fp16}")
    print(f"      Seed                : {seed}")

    training_args = TrainingArguments(
        output_dir=checkpoint_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        num_train_epochs=epochs,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=logging_steps,
        save_total_limit=1,
        fp16=use_fp16,
        report_to=[],
        seed=seed,
    )

    return training_args

In [ ]:
output_dir

In [ ]:
output_dir

In [ ]:
training_args = configure_finetuning_arguments(
    output_dir,
    learning_rate,
    train_batch_size,
    eval_batch_size,
    epochs,
    weight_decay,
    logging_steps,
    seed
)

training_args

#### Step 3: Build the Hugging Face Trainer

From the Hugging Face tutorial: `Trainer` combines the model, training configuration, tokenized datasets, padding logic, and metric function into one training object.

`DataCollatorWithPadding` dynamically pads every batch to the length of the longest sentence in that specific batch. This is more efficient than padding every sentence in the dataset to `max_length`.

`compute_metrics` tells `Trainer` to call `EvaluationMetric` after each evaluation cycle to calculate:

- Accuracy
- Precision
- Recall
- F1 Score

The `Trainer` receives:

| Input | Purpose |
|---|---|
| `model` | Pretrained BERT plus the binary classification head |
| `args` | `TrainingArguments` configuration |
| `train_dataset` | Tokenized training sentences and labels |
| `eval_dataset` | Tokenized validation sentences and labels |
| `tokenizer` | The tokenizer matched to the BERT checkpoint |
| `data_collator` | Dynamic batch-padding mechanism |
| `compute_metrics` | Function that computes validation classification metrics |

In [ ]:
# ============================================================
# BERT FINE-TUNING: STEP 3 — BUILD TRAINER
# ============================================================

def build_bert_trainer(
    model,
    training_args,
    train_dataset,
    val_dataset,
    tokenizer,
):
    """Combine model, datasets, tokenizer, metrics, and configuration."""
    print("\n[3/5] Building Trainer")
    print(f"      Train examples      : {len(train_dataset)}")
    print(f"      Validation examples : {len(val_dataset)}")
    print("      Data collator       : dynamic batch padding")
    print("      Compute metrics     : Accuracy, Precision, Recall, F1 Score")

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    return trainer

In [ ]:
trainer = build_bert_trainer(model,
    training_args,
    tokenized_train_dataset,
    tokenized_val_dataset,
    tokenizer
)

trainer

#### Step 4: Fine-Tune BERT


From D2L [1]: fine-tuning updates **all** pretrained Transformer encoder parameters while simultaneously training the newly added classification head.

The `[CLS]` token representation flows into the classification head. During BERT pretraining, the `[CLS]` representation is used by the NSP classifier because self-attention allows it to encode information from the complete BERT input sequence [1]. For this downstream sentence-classification task, the pretrained model’s sequence-level representation is used to predict the binary class.

During training, the Hugging Face `Trainer` prints information every `logging_steps` steps:

| Training Log | Meaning |
|---|---|
| `loss` | Cross-entropy loss on the current training batch |
| `grad_norm` | Gradient magnitude; very large spikes can indicate instability |
| `learning_rate` | Current learning-rate value; typically decays toward zero |
| `epoch` | Fractional progress through training; e.g., `1.5` means halfway through epoch 2 |

At the end of every epoch, the trainer prints validation metrics:

| Evaluation Log | Meaning |
|---|---|
| `eval_loss` | Cross-entropy loss on validation data |
| `eval_Accuracy` | Proportion of correct validation predictions |
| `eval_Precision` | Proportion of positive predictions that were correct |
| `eval_Recall` | Proportion of actual positive examples correctly found |
| `eval_F1 Score` | Harmonic mean of precision and recall |

In [ ]:
# ============================================================
# BERT FINE-TUNING: STEP 4 — FINE-TUNE
# ============================================================

def run_bert_finetuning(trainer, logging_steps):
    """Run supervised fine-tuning on the labeled training sentences."""
    print("\n[4/5] Starting fine-tuning...")
    print(f"      Logging frequency   : every {logging_steps} steps")
    print("      Training logs       : loss, grad_norm, learning_rate, epoch")
    print("      Validation logs     : eval_loss, Accuracy, Precision, Recall, F1 Score")

    trainer.train()

    return trainer

In [ ]:
print("train_dataset columns:", train_dataset.column_names)
print("tokenized_train_dataset columns:", tokenized_train_dataset.column_names)
print("tokenized train example:", tokenized_train_dataset[0])

In [ ]:
trainer = run_bert_finetuning(trainer, logging_steps)
trainer

#### Step 5: Save the Fine-Tuned Model and Tokenizer

The fine-tuned BERT model and its matching tokenizer are saved in `final_model`.

This saves the model weights, model configuration, classifier head, tokenizer vocabulary, and tokenizer configuration. The saved model can later be loaded for inference without retraining.

In [ ]:
def save_finetuned_bert(trainer, tokenizer, output_dir):
    """Save the fine-tuned model weights, configuration, and tokenizer."""
    final_model_dir = os.path.join(output_dir, "final_model")

    trainer.save_model(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)

    print("\n[5/5] Fine-tuning complete.")
    print(f"      ✓ Saved fine-tuned BERT model to : {final_model_dir}")
    print(f"      ✓ Saved tokenizer to             : {final_model_dir}")

    return final_model_dir

In [ ]:
save_finetuned_bert(trainer, tokenizer, output_dir)

In [ ]:
# # ============================================================
# # BERT FINE-TUNING: PIPELINE
# # ============================================================

# def finetune_bert_model(
#     train_dataset,
#     val_dataset,
#     model_name,
#     label_to_id,
#     id_to_label,
#     tokenizer,
#     output_dir
# ):
#     """Load, configure, fine-tune, and save a BERT sentence classifier."""
#     print("\n" + "=" * 40)
#     print("FINE-TUNE BERT")
#     print("=" * 40)

#     model = load_pretrained_bert_classifier(
#         model_name=model_name,
#         label_to_id=label_to_id,
#         id_to_label=id_to_label,
#     )

#     training_args = configure_finetuning_arguments(
#         output_dir=output_dir,
#         learning_rate=learning_rate,
#         train_batch_size=train_batch_size,
#         eval_batch_size=eval_batch_size,
#         epochs=epochs,
#         weight_decay=weight_decay,
#         logging_steps=logging_steps,
#         seed=seed
#     )

#     trainer = build_bert_trainer(
#         model=model,
#         training_args=training_args,
#         train_dataset=train_dataset,
#         val_dataset=val_dataset,
#         tokenizer=tokenizer,
#     )

#     trainer = run_bert_finetuning(
#         trainer=trainer,
#         logging_steps=logging_steps,
#     )

#     save_finetuned_bert(
#         trainer=trainer,
#         tokenizer=tokenizer,
#         output_dir=output_dir,
#     )

#     return trainer

In [ ]:
# finetuned_bert_model = finetune_bert_model(
#     tokenized_train_dataset,
#     tokenized_val_dataset,
#     model_name,
#     label_to_id,
#     id_to_label,
#     tokenizer,
#     output_dir
#     )

## Evaluation

In [ ]:
# ============================================================
# PREDICTION AND EVALUATION
# ============================================================

def softmax(logits):
    """Convert BERT classification logits into class probabilities."""
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted_logits)

    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

In [ ]:
def predict_dataset(trainer, tokenized_dataset):
    """Generate predicted labels and probabilities for all examples."""
    prediction_output = trainer.predict(tokenized_dataset)

    logits = prediction_output.predictions
    probabilities = softmax(logits)
    y_pred = np.argmax(probabilities, axis=1)

    return y_pred, probabilities

In [ ]:
# ============================================================
# STEP 1: GET GROUND-TRUTH LABELS
# ============================================================

def get_true_labels(source_df, label_id_column):
    """Extract integer ground-truth labels from the source DataFrame."""
    return source_df[label_id_column].to_numpy()

In [ ]:
y_true = get_true_labels(
    source_df=val_df,
    label_id_column=label_id_column,
)

print(f"Number of ground-truth labels: {len(y_true)}")
print(f"First 5 ground-truth labels: {y_true[:5]}")

In [ ]:
# ============================================================
# STEP 2: GET BERT PREDICTIONS AND PROBABILITIES
# ============================================================

def get_bert_predictions(trainer, tokenized_dataset):
    """Generate BERT predicted labels and probability vectors."""
    return predict_dataset(
        trainer=trainer,
        tokenized_dataset=tokenized_dataset,
    )

In [ ]:
y_pred, probabilities = get_bert_predictions(
    trainer=trainer,
    tokenized_dataset=tokenized_val_dataset,
)

print(f"Number of BERT predictions: {len(y_pred)}")
print(f"First 5 predicted labels: {y_pred[:5]}")
print(f"Probability matrix shape: {probabilities.shape}")
print(f"First probability vector: {probabilities[0]}")

In [ ]:
# ============================================================
# STEP 3: EVALUATE BERT PREDICTIONS
# ============================================================

def evaluate_predictions(y_true, y_pred, probabilities, split_name, seed):
    """Use EvaluationMetric to evaluate binary BERT predictions."""
    print("\n" + "=" * 40)
    print(f"EVALUATE: {split_name.upper()}")
    print("=" * 40)

    eval_report = EvaluationMetric.eval_classification_report(
        y_true,
        y_pred,
    )

    cm, tn, fp, fn, tp = EvaluationMetric.get_confusion_matrix(
        y_true,
        y_pred,
        by_category=True,
    )

    positive_class_probabilities = probabilities[:, 1]

    metrics_df = pd.DataFrame([{
        "seed": seed,
        "split": split_name,
        "accuracy": eval_report.get("accuracy"),
        "precision_class_0": eval_report.get("0", {}).get("precision"),
        "precision_class_1": eval_report.get("1", {}).get("precision"),
        "recall_class_0": eval_report.get("0", {}).get("recall"),
        "recall_class_1": eval_report.get("1", {}).get("recall"),
        "f1_class_0": eval_report.get("0", {}).get("f1-score"),
        "f1_class_1": eval_report.get("1", {}).get("f1-score"),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "roc_auc": EvaluationMetric.get_roc_auc(
            y_true,
            positive_class_probabilities,
        ),
        "pr_auc": EvaluationMetric.get_pr_auc(
            y_true,
            positive_class_probabilities,
        ),
    }])

    print(f"Confusion Matrix:\n{cm}\n")
    print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")

    return metrics_df, cm, eval_report

In [ ]:
split_name = "validation"

metrics_df, cm, report_dict = evaluate_predictions(
    y_true=y_true,
    y_pred=y_pred,
    probabilities=probabilities,
    split_name=split_name,
    seed=seed,
)

print("\nMetrics DataFrame:")
print(metrics_df)

print("\nConfusion Matrix:")
print(cm)

In [ ]:
# ============================================================
# STEP 4: CREATE OUTPUT DIRECTORY FOR THIS SPLIT
# ============================================================

def create_split_output_directory(output_dir, split_name):
    """Create directory for validation, test, or external evaluation outputs."""
    split_dir = os.path.join(output_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)
    return split_dir

In [ ]:
split_dir = create_split_output_directory(
    output_dir=output_dir,
    split_name=split_name,
)

print(f"Split output directory: {split_dir}")

## Log Experiment

In [ ]:
# ============================================================
# STEP 5: CREATE PREDICTIONS DATAFRAME
# ============================================================

def create_bert_results_dataframe(
    source_df,
    y_pred,
    probabilities,
    id_to_label,
):
    """Attach BERT predictions and probabilities to source data."""
    results_df = source_df.copy()

    results_df["BERT Predicted Label ID"] = y_pred
    results_df["BERT Predicted Label"] = [
        id_to_label[prediction]
        for prediction in y_pred
    ]

    results_df["BERT Probability Class 0"] = probabilities[:, 0]
    results_df["BERT Probability Class 1"] = probabilities[:, 1]

    return results_df


In [ ]:
predictions_df = create_bert_results_dataframe(
    source_df=val_df,
    y_pred=y_pred,
    probabilities=probabilities,
    id_to_label=id_to_label,
)

predictions_df.head(5)

In [ ]:
DataProcessing.save_to_file(
    predictions_df,
    path=split_dir,
    prefix="bert_predictions",
    save_file_type="csv",
    include_version=False,
)
print(f"Saved: {os.path.join(split_dir, 'bert_predictions.csv')}")

In [ ]:
DataProcessing.save_to_file(
    metrics_df,
    path=split_dir,
    prefix="bert_metrics_summary",
    save_file_type="csv",
    include_version=False,
)

In [ ]:
# ============================================================
# STEP 8: CREATE READABLE CONFUSION MATRIX DATAFRAME
# ============================================================

def create_confusion_matrix_dataframe(cm, id_to_label):
    """Add readable actual and predicted labels to the confusion matrix."""
    return pd.DataFrame(
        cm,
        index=[
            f"Actual: {id_to_label[0]}",
            f"Actual: {id_to_label[1]}",
        ],
        columns=[
            f"Predicted: {id_to_label[0]}",
            f"Predicted: {id_to_label[1]}",
        ],
    )

In [ ]:
confusion_matrix_df = create_confusion_matrix_dataframe(
    cm=cm,
    id_to_label=id_to_label,
)

print(confusion_matrix_df)

In [ ]:
# ============================================================
# STEP 9: SAVE CONFUSION MATRIX
# ============================================================

DataProcessing.save_to_file(
    confusion_matrix_df,
    path=split_dir,
    prefix="bert_confusion_matrix",
    save_file_type="csv",
    include_version=False,
)